# `odia_eval` — Capability Walkthrough

A self-contained tour of everything the [`odia_eval`](https://github.com/tripathysagar/odia_eval) library can do, runnable on **CPU in seconds** (no model, no GPU).

For a full evaluation against a real 4-bit quantised model see [`eval_odiabench.ipynb`](./eval_odiabench.ipynb) in this same folder.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tripathysagar/odia_eval/blob/main/notebooks/capability.ipynb)

**What this notebook covers**

1. Install & imports
2. Listing & loading benchmarks
3. Inspecting a sample row from each benchmark
4. Building Odia-language prompts
5. Scoring predictions (extractor demo per benchmark)
6. Special cases: TruthfulQA shuffling, ARC `E`, Odia digits, prompt-echo
7. `run_eval` end-to-end with a deterministic dummy generator
8. `run_all` + `full_report` + `to_records` export
9. Wilson CIs + chance baselines
10. Save / load results (JSONL round-trip)
11. Inspect failure modes
12. Reproducibility check (same seed → same rows)

In [1]:
%%capture
# Install odia_eval from GitHub (pure-stdlib, fast install).
# Also clones OdiaBench for the dataset/data/*.jsonl files.
import subprocess, sys, os, pathlib

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-qqq",
     "odia_eval @ git+https://github.com/tripathysagar/odia_eval.git"],
    check=True,
)

# Clone OdiaBench (for the benchmark data files) — skips if already present.
if "google.colab" in sys.modules:
    _data_root = pathlib.Path("/content/OdiaBench")
elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    _data_root = pathlib.Path("/kaggle/working/OdiaBench")
else:
    # Local: walk up from cwd to find the repo root
    _data_root = next(
        (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
         if (p / "dataset" / "data").exists()),
        None,
    )

if _data_root is None or not _data_root.exists():
    _data_root = pathlib.Path("OdiaBench")
    if not _data_root.exists():
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/tripathysagar/OdiaBench.git", str(_data_root)],
            check=True,
        )

DATA_DIR = str(_data_root / "dataset" / "data")

## 1 — Verify install

`odia_eval` has **zero required dependencies** — it's pure stdlib. The cell above installed it from [GitHub](https://github.com/tripathysagar/odia_eval) and cloned the OdiaBench data repo.

In [2]:
import odia_eval

print(f"odia_eval {odia_eval.__version__}")
print(f"public API: {len(odia_eval.__all__)} symbols")
print(f"data dir:   {DATA_DIR}")

odia_eval 0.2.0
public API: 23 symbols
data dir:   /home/sagar/git/OdiaBench/dataset/data


## 2 — Available benchmarks

`BENCHMARKS` is the canonical list of benchmark keys accepted everywhere in the library.

In [3]:
from odia_eval import BENCHMARKS, load_benchmark

print("Benchmarks:", BENCHMARKS)
print()
print(f"{'name':<12s} {'rows':>6s}  {'first id':>10s}")
print("-" * 32)
for name in BENCHMARKS:
    rows = load_benchmark(name, data_dir=DATA_DIR)
    print(f"{name:<12s} {len(rows):>6d}  {rows[0]['id']:>10}")

Benchmarks: ['arc', 'gsm8k', 'hellaswag', 'truthfulqa', 'winogrande']

name           rows    first id
--------------------------------
arc             299           0
gsm8k          1319           0


hellaswag     10042           0
truthfulqa      817           0
winogrande     1267           0


## 3 — Inspect one row from each benchmark

Every row has at minimum `id`, `question`, `answer`, `odia_question`, `odia_answer`. The extra fields are benchmark-specific (`mc1_choices`, `option1/option2`, `all_endings`, `answerKey`, `label`, …).

In [4]:
import textwrap

def _short(s, n=120):
    s = str(s).replace("\n", " ")
    return s if len(s) <= n else s[:n] + "…"

for name in BENCHMARKS:
    row = load_benchmark(name, n_samples=1, seed=0, data_dir=DATA_DIR)[0]
    print(f"── {name} ──  fields: {sorted(row.keys())}")
    for k, v in row.items():
        print(f"  {k:<16s} {_short(v, 100)}")
    print()

── arc ──  fields: ['answer', 'answerKey', 'id', 'odia_answer', 'odia_question', 'question', 'raw_id']
  id               197
  question         The element krypton is a gas that shows almost no chemical activity. To find another element with si…
  answer           A: an element in the same group B: an element in the same period C: an element with the same net cha…
  odia_question    ଉପାଦାନ କ୍ରିପ୍ଟନ୍ ଏକ ଗ୍ୟାସ୍ ଯାହା ପ୍ରାୟ କୌଣସି ରାସାୟନିକ କ୍ରିୟାଶୀଳତା ଦେଖାଏ ନାହିଁ। ସମାନ ଗୁଣ ସହିତ ଅନ୍ୟ ଉପା…
  odia_answer      A: ସମାନ ଗୋଷ୍ଠୀର ଏକ ଉପାଦାନ B: ସମାନ ପର୍ଯ୍ୟାୟର ଏକ ଉପାଦାନ C: ସମାନ ନିଷ୍ପତ୍ତି ଆଧାର ସହିତ ଏକ ଉପାଦାନ D: ସମାନ …
  raw_id           Mercury_7026198
  answerKey        A

── gsm8k ──  fields: ['answer', 'id', 'odia_answer', 'odia_question', 'question']
  id               788
  question         John is raising money for a school trip. He has applied for help from the school, which has decided …
  answer           John's school has decided to pay 300 / 2 = $<<300/2=150>>150 for his trip. In total John

── hellaswag ──  fields: ['activity_label', 'all_endings', 'answer', 'id', 'label', 'odia_answer', 'odia_question', 'question', 'raw_id']
  id               6311
  question         [header] How to double a recipe [title] Write each of the ingredients on a piece of paper. [step] Ch…
  answer           [substeps] If you have a copier, you may want to copy the original recipe and write in the margins, …
  odia_question    [ହେଡର୍] ଏକ ରେସିପି କିପରି ଦ୍ୱିଗୁଣିତ ହେବ [ଆଖ୍ୟା] ପ୍ରତ୍ୟେକ ଉପାଦାନକୁ କାଗଜ ଖଣ୍ଡରେ ଲେଖ | [ଷ୍ଟେପ୍] ରୋଷେୟାମାନ…
  odia_answer      [ସବଷ୍ଟେପ୍] ଯଦି ଆପଣଙ୍କର ଏକ କପିଅର୍ ଅଛି, ତେବେ ଆପଣ ମୂଳ ରେସିପି କପି କରି ମାର୍ଜିନରେ ଲେଖିବାକୁ ଚାହିଁପାରନ୍ତି, ଯ…
  raw_id           22823
  activity_label   Food and Entertaining
  all_endings      ['[substeps] If you have a copier, you may want to copy the original recipe and write in the margins…
  label            0

── truthfulqa ──  fields: ['answer', 'id', 'mc1_choices', 'mc1_labels', 'odia_answer', 'odia_question', 'question']
  id               394
  ques

## 4 — Build Odia-language prompts

`build_prompt(name, row)` is the only thing your `generate_fn` will be asked to consume. Each benchmark has its own template (CoT for GSM8K, MCQ blocks for ARC/TruthfulQA/HellaSwag, numbered choices for Winogrande).

In [5]:
from odia_eval import build_prompt

for name in BENCHMARKS:
    row = load_benchmark(name, n_samples=1, seed=0, data_dir=DATA_DIR)[0]
    prompt = build_prompt(name, row)
    print(f"══════════ {name} (row id {row['id']}) ══════════")
    print(prompt)
    print()

══════════ arc (row id 197) ══════════
ନିମ୍ନ ବିଜ୍ଞାନ ପ୍ରଶ୍ନ ପାଇଁ ସଠିକ ଉତ୍ତର ବାଛ (A, B, C ବା D)।

ପ୍ରଶ୍ନ: ଉପାଦାନ କ୍ରିପ୍ଟନ୍ ଏକ ଗ୍ୟାସ୍ ଯାହା ପ୍ରାୟ କୌଣସି ରାସାୟନିକ କ୍ରିୟାଶୀଳତା ଦେଖାଏ ନାହିଁ। ସମାନ ଗୁଣ ସହିତ ଅନ୍ୟ ଉପାଦାନ ଖୋଜିବା ପାଇଁ, ଛାତ୍ରକୁ ଉପାଦାନର ପର୍ଯ୍ୟାୟକ୍ରମ ତାଲିକାରେ କ'ଣ ଖୋଜିବାକୁ ପଡ଼ିବ?

A: ସମାନ ଗୋଷ୍ଠୀର ଏକ ଉପାଦାନ
B: ସମାନ ପର୍ଯ୍ୟାୟର ଏକ ଉପାଦାନ
C: ସମାନ ନିଷ୍ପତ୍ତି ଆଧାର ସହିତ ଏକ ଉପାଦାନ
D: ସମାନ ପରମାଣୁ ଭାର ସହିତ ଏକ ଉପାଦାନ

ସଠିକ ଉତ୍ତର:

══════════ gsm8k (row id 788) ══════════
ନିମ୍ନ ଗଣିତ ପ୍ରଶ୍ନଟି ପଢ଼ ଏବଂ ଧାପ ଧାପ ଭାବରେ ଉତ୍ତର ଦିଅ।
ଆପଣଙ୍କ ଉତ୍ତର ଶେଷ ଲାଇନରେ #### N (N = ଉତ୍ତର ସଂଖ୍ୟା) ଆକାରରେ ସାରନ୍ତୁ।

ପ୍ରଶ୍ନ: ଜନ୍ ବିଦ୍ୟାଳୟ ଯାତ୍ରା ପାଇଁ ଟଙ୍କା ସଂଗ୍ରହ କରୁଛନ୍ତି। ବିଦ୍ୟାଳୟ ଯାତ୍ରା ଖର୍ଚ୍ଚର ଅଧା ଦେବାକୁ ନିଷ୍ପତ୍ତି ନେଇଛି। ଯାତ୍ରା $300 ଏବଂ ଜନ୍‌ଙ୍କ ପାଖରେ $50 ଥିଲେ, ଆଉ କେତେ ଟଙ୍କା ଲାଗିବ?
ଉତ୍ତର:



══════════ hellaswag (row id 6311) ══════════
ନିମ୍ନ ଅନୁଚ୍ଛେଦ ପାଇଁ ସବୁଠୁ ଉପଯୁକ୍ତ ସମାପ୍ତି ବାଛ (A, B, C ବା D)।

ପ୍ରସଙ୍ଗ: [ହେଡର୍] ଏକ ରେସିପି କିପରି ଦ୍ୱିଗୁଣିତ ହେବ [ଆଖ୍ୟା] ପ୍ରତ୍ୟେକ ଉପାଦାନକୁ କାଗଜ ଖଣ୍ଡରେ ଲେଖ | [ଷ୍ଟେପ୍] ରୋଷେୟାମାନେ ଆପଣଙ୍କ ମୁଣ୍ଡରେ ଏକ ରେସିପି ମାପିବା ପାଇଁ ପରାମର୍ଶ ଦିଅନ୍ତି ନାହିଁ | ଆପଣ ଆବଶ୍ୟକ କରୁଥିବା ପରିମାଣକୁ ସମୟ ପୂର୍ବରୁ ଲେଖିବା ଉଚିତ୍ |

A: [substeps] If you have a copier, you may want to copy the original recipe and write in the margins, so that you have the instructions next to the ingredients. [title] Write down all of the vegetables, flour and meat products in 1 column.
B: [substeps] To " double " a recipe, you need to produce a detailed ingredient chart that you can use during the process. Add together the ingredient base, texture and-if needed-a hacksaw.
C: Having a thin target circle in your head allows to create a more balanced meal. [substeps] Include between 1.5 and 2.5 oz.
D: Choose a single side of the recipe and put your cursor in the appropriate place. [substeps] Write out t

## 5 — Score a single prediction

`score(name, prediction, row)` returns a `ScoreResult(correct, extracted, gold, prediction)`. The extractors are deliberately lenient — they search the whole string so chain-of-thought and short answers both work.

In [6]:
from odia_eval import score

demos = {
    "gsm8k":      "ଚିନ୍ତା… #### 18",
    "arc":        "ସଠିକ ଉତ୍ତର D ଅଟେ।",
    "truthfulqa": "Answer: B",
    "winogrande": "ଉତ୍ତର ୨",   # Odia digit 2
    "hellaswag":  "The best ending is D.",
}

for name in BENCHMARKS:
    row = load_benchmark(name, n_samples=1, seed=0, data_dir=DATA_DIR)[0]
    sr = score(name, demos[name], row)
    print(f"{name:<12s} gold={sr.gold!r:<6s} extracted={sr.extracted!r:<6s} correct={sr.correct}")

arc          gold='A'    extracted='D'    correct=False
gsm8k        gold='100'  extracted='18'   correct=False


hellaswag    gold='A'    extracted='D'    correct=False
truthfulqa   gold='C'    extracted='B'    correct=False
winogrande   gold='1'    extracted='2'    correct=False


## 6 — Special cases worth knowing

### 6a. TruthfulQA choices are shuffled by row id

The upstream HF dataset stores the correct answer at index 0 in every row. Without shuffling, gold would always be `"A"` and the benchmark would be trivially gamed. `build_prompt_truthfulqa` runs a deterministic permutation seeded by `row["id"]`; `score_truthfulqa` replays the same permutation. You can read the permutation directly via `truthfulqa_permutation`.

In [7]:
from collections import Counter
from odia_eval import truthfulqa_permutation

tfqa = load_benchmark("truthfulqa", data_dir=DATA_DIR)
gold_dist = Counter(score("truthfulqa", "", r).gold for r in tfqa)
print(f"TruthfulQA gold distribution over all {len(tfqa)} rows:")
for letter, n in sorted(gold_dist.items()):
    print(f"  {letter}: {n}")

print()
print("permutation(row_id=0, n=4):", truthfulqa_permutation(0, 4))
print("permutation(row_id=1, n=4):", truthfulqa_permutation(1, 4))
print("permutation(row_id=0, n=4) again (determinism):", truthfulqa_permutation(0, 4))

TruthfulQA gold distribution over all 817 rows:
  A: 186
  B: 192
  C: 177
  D: 130
  E: 64
  F: 31
  G: 17
  H: 8
  I: 7
  J: 4
  M: 1

permutation(row_id=0, n=4): [2, 1, 3, 0]
permutation(row_id=1, n=4): [1, 2, 3, 0]
permutation(row_id=0, n=4) again (determinism): [2, 1, 3, 0]


### 6b. Prompt-echo is handled automatically

Many `generate_fn` implementations return `prompt + completion` (e.g. `tokenizer.batch_decode(model.generate(...))` without slicing off input ids). The MCQ prompts contain `"A: ..."`, `"1: ..."` etc., which would otherwise win against the extractors. `run_eval` calls `_strip_prompt(pred, prompt)` as a safety net.

In [8]:
from odia_eval import _strip_prompt

row = load_benchmark("arc", n_samples=1, seed=0, data_dir=DATA_DIR)[0]
prompt = build_prompt("arc", row)

# Simulate a tokenizer that echoes the prompt AND prepends a BOS marker
echoed = "<bos>" + prompt + " D"

print("Raw decoded length :", len(echoed))
print("Stripped completion:", repr(_strip_prompt(echoed, prompt).strip()))
print()
print("Scoring the raw echoed prediction directly (no run_eval) would now")
print("still work because we ALSO normalise inside the scorer, but the")
print("naive A-first regex used to extract the first 'A:' from the prompt.")

Raw decoded length : 387
Stripped completion: 'D'

Scoring the raw echoed prediction directly (no run_eval) would now
still work because we ALSO normalise inside the scorer, but the
naive A-first regex used to extract the first 'A:' from the prompt.


### 6c. Odia digits and number-words in Winogrande

The Winogrande scorer translates Odia numerals (`୦`–`୯`) and the common number-words (`ଗୋଟିଏ`/`ଏକ` → 1, `ଦୁଇ`/`ଦୁଇଟି` → 2) before extraction.

In [9]:
fake_row = {"answer_idx": "2"}
for pred in ["2", "୨", "ଉତ୍ତର ଦୁଇ", "ଦ୍ୱିତୀୟ ବିକଳ୍ପ", "I think 2 is correct"]:
    sr = score("winogrande", pred, fake_row)
    print(f"  {pred!r:<35s} → extracted={sr.extracted!r} correct={sr.correct}")

  '2'                                 → extracted='2' correct=True
  '୨'                                 → extracted='2' correct=True
  'ଉତ୍ତର ଦୁଇ'                         → extracted='2' correct=True
  'ଦ୍ୱିତୀୟ ବିକଳ୍ପ'                    → extracted='2' correct=True
  'I think 2 is correct'              → extracted='2' correct=True


## 7 — `run_eval` with a dummy generator

`run_eval(benchmark, generate_fn, ...)` is the workhorse. Here we plug in two no-model baselines so the notebook runs on CPU:

* **`always_A`** — outputs `"A"` for every prompt. Useful for sanity-checking the gold distribution (a well-shuffled MCQ benchmark should hover around `1/n_choices`).
* **`random_baseline`** — outputs a uniformly random valid token for each benchmark. Should land near random-chance accuracy.

In [10]:
import random
from odia_eval import run_eval, score_report

def always_A(prompts):
    return ["A"] * len(prompts)

rng = random.Random(0)
def random_baseline_for(benchmark):
    # Each benchmark has its own answer vocabulary
    vocab = {
        "gsm8k":      [f"#### {i}" for i in range(0, 200)],
        "arc":        list("ABCD"),
        "truthfulqa": list("ABCDEFGHIJKLM"),
        "winogrande": ["1", "2"],
        "hellaswag":  list("ABCD"),
    }[benchmark]
    def _gen(prompts):
        return [rng.choice(vocab) for _ in prompts]
    return _gen

N = 100          # small sample to keep this notebook fast
BATCH = 64       # library default — same value works on T4 / L4 / A100 / CPU here

print("── always_A baseline ──")
for name in BENCHMARKS:
    res = run_eval(name, always_A, n_samples=N, seed=0, batch_size=BATCH, data_dir=DATA_DIR, verbose=False)
    print(" ", score_report(res))

print("\n── random baseline ──")
for name in BENCHMARKS:
    res = run_eval(name, random_baseline_for(name), n_samples=N, seed=0, batch_size=BATCH, data_dir=DATA_DIR, verbose=False)
    print(" ", score_report(res))

── always_A baseline ──
  arc            17/100    17.0%  [ 10.9– 25.5]  chance 25.1%  Δ  -8.1
  gsm8k           0/100     0.0%  [  0.0–  3.7]  chance  0.0%  Δ  +0.0


  hellaswag      21/100    21.0%  [ 14.2– 30.0]  chance 25.0%  Δ  -4.0
  truthfulqa     28/100    28.0%  [ 20.1– 37.5]  chance 22.6%  Δ  +5.4
  winogrande      0/100     0.0%  [  0.0–  3.7]  chance 50.0%  Δ -50.0

── random baseline ──
  arc            29/100    29.0%  [ 21.0– 38.5]  chance 25.1%  Δ  +3.9
  gsm8k           2/100     2.0%  [  0.6–  7.0]  chance  0.0%  Δ  +2.0


  hellaswag      22/100    22.0%  [ 15.0– 31.1]  chance 25.0%  Δ  -3.0
  truthfulqa      9/100     9.0%  [  4.8– 16.2]  chance 22.6%  Δ -13.6
  winogrande     47/100    47.0%  [ 37.5– 56.7]  chance 50.0%  Δ  -3.0


## 8 — `run_all` + `full_report` + `to_records`

`run_all` loops over every benchmark and returns `dict[str, list[EvalRow]]`. `full_report` prints a tidy accuracy table with a mean row; `to_records` flattens everything to a list of dicts for pandas / JSONL export.

In [11]:
from odia_eval import run_all, full_report, to_records

# Per-benchmark dispatch: random baseline tailored to each task
def smart_random(prompts, _bench=[None]):
    # run_all calls generate_fn one batch at a time per benchmark, so we
    # latch onto the current benchmark by sniffing the prompt header.
    head = prompts[0][:30]
    if "ଗଣିତ" in head:        bench = "gsm8k"
    elif "ବିଜ୍ଞାନ" in head:    bench = "arc"
    elif "ଶୂନ୍ୟସ୍ଥାନ" in head: bench = "winogrande"
    elif "ଅନୁଚ୍ଛେଦ" in head:   bench = "hellaswag"
    else:                       bench = "truthfulqa"
    return random_baseline_for(bench)(prompts)

all_results = run_all(smart_random, n_samples=100, seed=0, batch_size=BATCH, data_dir=DATA_DIR, verbose=False)
print(full_report(all_results))

print()
records = to_records(all_results)
print(f"to_records → {len(records)} flat dicts, first row:")
print(" ", records[0])

OdiaBench Evaluation Results  (95% Wilson CI)
arc            30/100    30.0%  [ 21.9– 39.6]  chance 25.1%  Δ  +4.9
gsm8k           1/100     1.0%  [  0.2–  5.4]  chance  0.0%  Δ  +1.0
hellaswag      28/100    28.0%  [ 20.1– 37.5]  chance 25.0%  Δ  +3.0
truthfulqa      8/100     8.0%  [  4.1– 15.0]  chance 22.6%  Δ -14.6
winogrande     45/100    45.0%  [ 35.6– 54.8]  chance 50.0%  Δ  -5.0
------------------------------------------------------------
MEAN                     22.4%                                 Δ  -2.1

to_records → 500 flat dicts, first row:
  {'benchmark': 'arc', 'id': 197, 'correct': False, 'extracted': 'D', 'gold': 'A', 'prediction': 'D'}


### Wilson confidence intervals + chance deltas

`full_report` now shows a 95% Wilson CI and the gap above the known chance baseline (`CHANCE_LEVELS`). The random baseline above should mostly land inside ±CI of its chance level — a useful sanity check that the eval pipeline isn't biased.

In [12]:
from odia_eval import wilson_ci, chance_level, CHANCE_LEVELS, accuracy_with_ci

print("CHANCE_LEVELS (theoretical random-baseline accuracy):")
for name, p in CHANCE_LEVELS.items():
    print(f"  {name:<12s} {p*100:5.1f}%")
print()
print("Wilson CI at small n:")
print(f"  20/100  → {wilson_ci(20, 100)}  (±~8.4 pp — why n=200 matters)")
print(f"  20/1000 → {wilson_ci(20, 1000)}  (±~1.3 pp at 10× the data)")
print()
print("accuracy_with_ci on the ARC random baseline:")
acc, lo, hi, k, n = accuracy_with_ci(all_results['arc'])
print(f"  {k}/{n}  acc={acc*100:.1f}%  CI=[{lo*100:.1f}, {hi*100:.1f}]  chance={chance_level('arc')*100:.1f}%")

CHANCE_LEVELS (theoretical random-baseline accuracy):
  gsm8k          0.0%
  arc           25.1%
  truthfulqa    22.6%
  winogrande    50.0%
  hellaswag     25.0%

Wilson CI at small n:
  20/100  → (0.1333669333310325, 0.28882916559315885)  (±~8.4 pp — why n=200 matters)
  20/1000 → (0.01298368291641514, 0.03069000522971778)  (±~1.3 pp at 10× the data)

accuracy_with_ci on the ARC random baseline:
  30/100  acc=30.0%  CI=[21.9, 39.6]  chance=25.1%


### Persist & reload results (no need to re-run generation to re-score)

`save_results(all_results, path)` writes one JSONL row per evaluated example (benchmark, id, prompt, gold, extracted, prediction, correct). `load_results(path)` reconstructs the original `dict[str, list[EvalRow]]` so every downstream helper (`full_report`, `show_failures`, `to_records`) still works.

In [13]:
import tempfile, os
from odia_eval import save_results, load_results

with tempfile.NamedTemporaryFile("w", suffix=".jsonl", delete=False) as fh:
    tmp = fh.name

n_written = save_results(all_results, tmp)
size_kb = os.path.getsize(tmp) / 1024
print(f"save_results → {n_written} rows, {size_kb:.1f} KB at {tmp}")

reloaded = load_results(tmp)
print(f"load_results → {sum(len(v) for v in reloaded.values())} rows across {list(reloaded)}")
print()
print("Reports match after round-trip:", full_report(all_results) == full_report(reloaded))

os.unlink(tmp)

save_results → 500 rows, 454.1 KB at /tmp/tmp3gvewmnc.jsonl
load_results → 500 rows across ['arc', 'gsm8k', 'hellaswag', 'truthfulqa', 'winogrande']

Reports match after round-trip: True


### Inspect failure modes

`show_failures(results, n=10)` prints the prompt, gold, extracted answer, and raw prediction for the first N wrong rows — exactly what you need to decide if the model was wrong or the extractor was wrong.

In [14]:
from odia_eval import show_failures

print("First 3 ARC failures:\n")
show_failures(all_results["arc"], n=3, max_chars=120)

First 3 ARC failures:

── arc #197 ──
  prompt:     ନିମ୍ନ ବିଜ୍ଞାନ ପ୍ରଶ୍ନ ପାଇଁ ସଠିକ ଉତ୍ତର ବାଛ (A, B, C ବା D)।  ପ୍ରଶ୍ନ: ଉପାଦାନ କ୍ରିପ୍ଟନ୍ ଏକ ଗ୍ୟାସ୍ ଯାହା ପ୍ରାୟ କୌଣସି ରାସାୟନିକ କ…
  gold:       'A'
  extracted:  'D'
  prediction: D

── arc #215 ──
  prompt:     ନିମ୍ନ ବିଜ୍ଞାନ ପ୍ରଶ୍ନ ପାଇଁ ସଠିକ ଉତ୍ତର ବାଛ (A, B, C ବା D)।  ପ୍ରଶ୍ନ: ଉତ୍ପାଦକମାନେ ଯେଉଁ ଗ୍ୟାସ୍ ମୁକ୍ତ କରନ୍ତି ଯାହାକୁ ଉପଭୋକ୍ତାମା…
  gold:       'A'
  extracted:  'D'
  prediction: D

── arc #20 ──
  prompt:     ନିମ୍ନ ବିଜ୍ଞାନ ପ୍ରଶ୍ନ ପାଇଁ ସଠିକ ଉତ୍ତର ବାଛ (A, B, C ବା D)।  ପ୍ରଶ୍ନ: ଜୈବିକ ବିବର୍ତ୍ତନ ଏହି ସବୁ ମାଧ୍ୟମରେ ଘଟିପାରେ ବ୍ୟତୀତ  A: ପ୍…
  gold:       'B'
  extracted:  'A'
  prediction: A



### Optional: pandas export

Requires `pandas` (listed in `odia_eval/requirements.txt` as an optional extra).

In [15]:
try:
    import pandas as pd
    df = pd.DataFrame(records)
    print(df.head())
    print()
    print("Accuracy per benchmark (pandas):")
    print(df.groupby("benchmark")["correct"].mean().mul(100).round(1).astype(str) + " %")
except ImportError:
    print("pandas not installed — `pip install -r odia_eval/requirements.txt` to enable.")

  benchmark   id  correct extracted gold prediction
0       arc  197    False         D    A          D
1       arc  215    False         D    A          D
2       arc   20    False         A    B          A
3       arc  132    False         A    D          A
4       arc  261    False         D    B          D

Accuracy per benchmark (pandas):
benchmark
arc           30.0 %
gsm8k          1.0 %
hellaswag     28.0 %
truthfulqa     8.0 %
winogrande    45.0 %
Name: correct, dtype: object


## 9 — Reproducibility check

Same `seed` → same row sample → same prompts → same predictions → same scores. This is what makes cross-model comparisons meaningful.

In [16]:
a = [r["id"] for r in load_benchmark("arc", n_samples=10, seed=42, data_dir=DATA_DIR)]
b = [r["id"] for r in load_benchmark("arc", n_samples=10, seed=42, data_dir=DATA_DIR)]
c = [r["id"] for r in load_benchmark("arc", n_samples=10, seed=43, data_dir=DATA_DIR)]

print("seed=42 (call 1):", a)
print("seed=42 (call 2):", b, "← identical" if a == b else "← DIFFERENT (bug!)")
print("seed=43        :", c, "← different sample" if a != c else "← same (bug!)")

seed=42 (call 1): [57, 12, 140, 125, 114, 71, 52, 279, 44, 216]
seed=42 (call 2): [57, 12, 140, 125, 114, 71, 52, 279, 44, 216] ← identical
seed=43        : [19, 146, 73, 236, 189, 49, 232, 255, 9, 263] ← different sample


## Next steps

* Swap `smart_random` / `always_A` for a real `generate_fn` wrapping your model.
* For a full 4-bit Gemma reference setup, open [`eval_odiabench.ipynb`](./eval_odiabench.ipynb) in this folder.
* Library repo + reference: [github.com/tripathysagar/odia\_eval](https://github.com/tripathysagar/odia_eval)
* Install with optional extras: `pip install "odia_eval[all] @ git+https://github.com/tripathysagar/odia_eval.git"`